# 03 - Plate Recognition Benchmark

Notebook ini mengukur akurasi pembacaan crop plat secara terpisah dari akurasi detector. EasyOCR dan PaddleOCR menerima crop serta preprocessing yang sama, lalu ensemble memilih kandidat berdasarkan agreement, confidence, dan format nomor polisi Indonesia.

## 1. Environment

Pilih kernel `.venv`. PaddleOCR 3 membutuhkan `opencv-contrib-python`. Untuk environment lama, tutup semua kernel Jupyter terlebih dahulu, jalankan perintah migrasi OpenCV pada cell berikut dari terminal VS Code, lalu restart kernel.

In [ ]:
# Jalankan di terminal dengan kernel Jupyter berhenti jika environment dibuat sebelum notebook ini:
# python -m pip uninstall -y opencv-python opencv-python-headless
# python -m pip install opencv-contrib-python
# python -m pip install paddlepaddle==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cpu/
# python -m pip install paddleocr


In [ ]:
import re
from collections import Counter
from difflib import SequenceMatcher
from pathlib import Path
from time import perf_counter

import cv2
import easyocr
import ipywidgets as widgets
import numpy as np
import pandas as pd
import torch
from IPython.display import Image as DisplayImage, clear_output, display
from paddleocr import TextRecognition
from ultralytics import YOLO

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'ml').is_dir() and (candidate / 'data').is_dir():
            return candidate
    raise FileNotFoundError('Project root tidak ditemukan. Buka notebook dari repository Smart Parking System.')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
MODEL_PATH = PROJECT_ROOT / 'ml/experiments/yolo_license_plate_detector/yolo11n_plate_v1/weights/best.pt'
LABELED_TEST_DIR = PROJECT_ROOT / 'data/raw/indonesian-plate-number/plate_number_recognition_datasets_v1/images/test'
OUTPUT_DIR = PROJECT_ROOT / 'ml/experiments/plate_recognition_benchmark'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print(f'Project: {PROJECT_ROOT}')
print(f'Device: {DEVICE}')


## 2. Load models

In [ ]:
if not MODEL_PATH.is_file():
    raise FileNotFoundError(f'YOLO checkpoint tidak ditemukan: {MODEL_PATH}')

detector = YOLO(str(MODEL_PATH))
easy_reader = easyocr.Reader(['en'], gpu=torch.cuda.is_available())
paddle_reader = TextRecognition(model_name='en_PP-OCRv5_mobile_rec', device='gpu:0' if torch.cuda.is_available() else 'cpu')
print('YOLO, EasyOCR, dan PaddleOCR siap.')


## 3. Recognition helpers

In [ ]:
OCR_ALLOWLIST = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'
PLATE_PATTERN = re.compile(r'^[A-Z]{1,3}[0-9]{1,4}[A-Z]{0,3}$')

def normalize_plate_text(value: str) -> str:
    return re.sub(r'[^A-Z0-9]', '', str(value).upper())

def character_similarity(prediction: str, ground_truth: str) -> float:
    return SequenceMatcher(None, normalize_plate_text(prediction), normalize_plate_text(ground_truth)).ratio()

def plate_preprocessing_variants(crop: np.ndarray) -> list[tuple[str, np.ndarray]]:
    scale = max(1.0, 112 / max(crop.shape[0], 1))
    resized = cv2.resize(crop, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    gray = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY) if resized.ndim == 3 else resized
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)
    _, otsu = cv2.threshold(clahe, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    adaptive = cv2.adaptiveThreshold(clahe, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 7)
    return [('original', resized), ('clahe', clahe), ('otsu', otsu), ('adaptive', adaptive)]

def candidate_score(text: str, confidence: float) -> float:
    score = float(confidence)
    score += 0.25 if PLATE_PATTERN.fullmatch(text) else 0.0
    score += 0.05 if any(c.isalpha() for c in text) and any(c.isdigit() for c in text) else 0.0
    return score - (0.20 if not 3 <= len(text) <= 10 else 0.0)

def read_easyocr_variant(image: np.ndarray) -> tuple[str, float]:
    detections = easy_reader.readtext(image, detail=1, paragraph=False, allowlist=OCR_ALLOWLIST)
    if not detections:
        return '', 0.0
    ordered = sorted(detections, key=lambda item: min(point[0] for point in item[0]))
    text = normalize_plate_text(''.join(item[1] for item in ordered))
    confidence = float(np.mean([item[2] for item in ordered]))
    return text, confidence

def read_paddle_variant(image: np.ndarray) -> tuple[str, float]:
    if image.ndim == 2:
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    elif image.shape[2] == 4:
        image = cv2.cvtColor(image, cv2.COLOR_BGRA2BGR)
    image = np.ascontiguousarray(image, dtype=np.uint8)
    result = list(paddle_reader.predict(input=image, batch_size=1))
    if not result:
        return '', 0.0
    payload = result[0].json.get('res', result[0].json)
    return normalize_plate_text(payload.get('rec_text', '')), float(payload.get('rec_score', 0.0))

def recognize_crop(crop: np.ndarray) -> tuple[str, list[dict]]:
    candidates = []
    for variant_name, variant in plate_preprocessing_variants(crop):
        for engine, reader in [('easyocr', read_easyocr_variant), ('paddleocr', read_paddle_variant)]:
            started_at = perf_counter()
            text, confidence = reader(variant)
            candidates.append({'engine': engine, 'variant': variant_name, 'text': text, 'confidence': confidence, 'latency_ms': (perf_counter() - started_at) * 1000})
    votes = Counter(item['text'] for item in candidates if item['text'])
    for item in candidates:
        item['score'] = candidate_score(item['text'], item['confidence']) + 0.12 * max(0, votes[item['text']] - 1)
    winner = max(candidates, key=lambda item: item['score'], default={'text': ''})
    return winner['text'], candidates


## 4. Recognition benchmark with ground truth

Bagian ini sengaja memakai crop plat berlabel langsung. Tujuannya mengukur recognizer tanpa mencampur error dari bounding box YOLO.

In [ ]:
MAX_BENCHMARK_IMAGES = None  # None = semua crop pada split test.
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def ground_truth_from_filename(path: Path) -> str:
    match = re.match(r'^\d+\.(.+?)-\d{2}-\d{2}$', path.stem)
    return normalize_plate_text(match.group(1) if match else path.stem)

image_paths = sorted(path for path in LABELED_TEST_DIR.iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS)
if MAX_BENCHMARK_IMAGES is not None:
    image_paths = image_paths[:MAX_BENCHMARK_IMAGES]

benchmark_rows = []
for index, image_path in enumerate(image_paths, start=1):
    crop = cv2.imread(str(image_path))
    ground_truth = ground_truth_from_filename(image_path)
    ensemble_text, candidates = recognize_crop(crop)
    for engine in ('easyocr', 'paddleocr'):
        engine_candidates = [item for item in candidates if item['engine'] == engine]
        best = max(engine_candidates, key=lambda item: candidate_score(item['text'], item['confidence']))
        benchmark_rows.append({'file': image_path.name, 'ground_truth': ground_truth, 'method': engine, 'prediction': best['text'], 'variant': best['variant'], 'confidence': best['confidence'], 'exact_match': best['text'] == ground_truth, 'character_similarity': character_similarity(best['text'], ground_truth), 'latency_ms': sum(item['latency_ms'] for item in engine_candidates)})
    benchmark_rows.append({'file': image_path.name, 'ground_truth': ground_truth, 'method': 'ensemble', 'prediction': ensemble_text, 'variant': 'voting', 'confidence': max((item['confidence'] for item in candidates if item['text'] == ensemble_text), default=0.0), 'exact_match': ensemble_text == ground_truth, 'character_similarity': character_similarity(ensemble_text, ground_truth), 'latency_ms': sum(item['latency_ms'] for item in candidates)})
    print(f'[{index}/{len(image_paths)}] GT={ground_truth} | ensemble={ensemble_text}')

benchmark_df = pd.DataFrame(benchmark_rows)
display(benchmark_df)


In [ ]:
summary_df = benchmark_df.groupby('method', as_index=False).agg(exact_accuracy=('exact_match', 'mean'), average_character_similarity=('character_similarity', 'mean'), median_latency_ms=('latency_ms', 'median'))
summary_df[['exact_accuracy', 'average_character_similarity']] *= 100
display(summary_df.round(2))
display(benchmark_df[(benchmark_df['method'] == 'ensemble') & ~benchmark_df['exact_match']][['file', 'ground_truth', 'prediction', 'character_similarity']])
output_path = OUTPUT_DIR / 'recognition_benchmark.csv'
benchmark_df.to_csv(output_path, index=False)
print(f'Result saved: {output_path}')


## 5. Upload a full vehicle image

YOLO mencari bounding box plat, lalu setiap crop dibandingkan oleh kedua OCR. Isi GT bersifat opsional dan hanya dipakai untuk menghitung similarity.

In [ ]:
upload = widgets.FileUpload(accept='image/*', multiple=False, description='Choose image')
ground_truth_input = widgets.Text(description='GT (optional)', placeholder='L 1846 JE')
run_button = widgets.Button(description='Detect and read', button_style='primary')
upload_output = widgets.Output()

def crop_box_with_margin(image: np.ndarray, xyxy: list[float], margin: float = 0.08) -> np.ndarray:
    height, width = image.shape[:2]
    x1, y1, x2, y2 = xyxy
    pad_x, pad_y = (x2 - x1) * margin, (y2 - y1) * margin
    left, top = max(0, int(x1 - pad_x)), max(0, int(y1 - pad_y))
    right, bottom = min(width, int(x2 + pad_x)), min(height, int(y2 + pad_y))
    return image[top:bottom, left:right].copy()

def uploaded_bytes(widget: widgets.FileUpload) -> bytes:
    value = widget.value
    if isinstance(value, dict):
        return bytes(next(iter(value.values()))['content'])
    return bytes(value[0]['content'])

def run_uploaded_image(_):
    with upload_output:
        clear_output(wait=True)
        if not upload.value:
            print('Pilih satu gambar terlebih dahulu.')
            return
        image = cv2.imdecode(np.frombuffer(uploaded_bytes(upload), np.uint8), cv2.IMREAD_COLOR)
        result = detector.predict(image, conf=0.35, imgsz=960, device=DEVICE, verbose=False)[0]
        boxes = result.boxes if result.boxes is not None else []
        print(f'Detected plates: {len(boxes)}')
        for index, box in enumerate(boxes, start=1):
            crop = crop_box_with_margin(image, box.xyxy[0].tolist())
            prediction, candidates = recognize_crop(crop)
            ok, encoded = cv2.imencode('.jpg', crop)
            print(f"\nPlate {index}: {prediction or 'UNREADABLE'} | detection={float(box.conf[0]):.2%}")
            if ground_truth_input.value.strip():
                print(f'GT: {normalize_plate_text(ground_truth_input.value)} | similarity={character_similarity(prediction, ground_truth_input.value):.2%}')
            if ok:
                display(DisplayImage(data=encoded.tobytes(), width=500))
            display(pd.DataFrame(candidates)[['engine', 'variant', 'text', 'confidence', 'score', 'latency_ms']].sort_values('score', ascending=False).round(3))

run_button.on_click(run_uploaded_image)
display(widgets.VBox([upload, ground_truth_input, run_button, upload_output]))


## 6. Batch test on TEMPORARY datasets

`Test V1` diuji end-to-end (YOLO lalu OCR). `Test V2` memakai anotasi polygon untuk menghasilkan crop yang konsisten. Nama file V2 memuat ID sumber, bukan transkripsi nomor plat, sehingga kedua dataset memerlukan ground truth manual untuk menghitung akurasi OCR yang valid.

In [ ]:
TEST_V1_DIR = PROJECT_ROOT / 'TEMPORARY/Test V1'
TEST_V2_DIR = PROJECT_ROOT / 'TEMPORARY/Test V2'
BATCH_OUTPUT_DIR = OUTPUT_DIR / 'temporary_tests'
BATCH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_VARIANTS = {'original', 'clahe', 'otsu', 'adaptive'}
BATCH_ENGINES = {'paddleocr'}  # Tambahkan 'easyocr' hanya jika ingin membandingkan ulang.
TEST_V1_MAX_IMAGES = None     # Proses semua 100 gambar.
TEST_V2_SPLIT = 'test'         # Gunakan held-out test split terlebih dahulu.
TEST_V2_MAX_IMAGES = None      # Proses semua gambar pada split test.
CHECKPOINT_EVERY = 25

def recognize_crop_for_batch(crop: np.ndarray) -> tuple[str, list[dict]]:
    readers = {'easyocr': read_easyocr_variant, 'paddleocr': read_paddle_variant}
    candidates = []
    for variant_name, variant in plate_preprocessing_variants(crop):
        if variant_name not in BATCH_VARIANTS:
            continue
        for engine in BATCH_ENGINES:
            started_at = perf_counter()
            text, confidence = readers[engine](variant)
            candidates.append({'engine': engine, 'variant': variant_name, 'text': text, 'confidence': confidence, 'latency_ms': (perf_counter() - started_at) * 1000})
    votes = Counter(item['text'] for item in candidates if item['text'])
    for item in candidates:
        item['score'] = candidate_score(item['text'], item['confidence']) + 0.12 * max(0, votes[item['text']] - 1)
    winner = max(candidates, key=lambda item: item['score'], default={'text': ''})
    return winner['text'], candidates

def test_v2_reference_id(image_path: Path) -> str:
    source_name = image_path.name.split('_jpg.rf.')[0]
    match = re.search(r'\d{8}_\d+_(.+)$', source_name)
    return normalize_plate_text(match.group(1) if match else '')

def crop_from_yolo_polygon(image: np.ndarray, label_path: Path, margin: float = 0.05) -> np.ndarray | None:
    if not label_path.is_file():
        return None
    rows = [line.split() for line in label_path.read_text(encoding='utf-8').splitlines() if line.strip()]
    if not rows:
        return None
    values = [float(value) for value in rows[0][1:]]
    if len(values) < 4:
        return None
    height, width = image.shape[:2]
    if len(values) == 4:
        center_x, center_y, box_width, box_height = values
        x_values = [center_x - box_width / 2, center_x + box_width / 2]
        y_values = [center_y - box_height / 2, center_y + box_height / 2]
    else:
        x_values, y_values = values[0::2], values[1::2]
    x1, x2 = min(x_values) * width, max(x_values) * width
    y1, y2 = min(y_values) * height, max(y_values) * height
    pad_x, pad_y = (x2 - x1) * margin, (y2 - y1) * margin
    left, right = max(0, int(x1 - pad_x)), min(width, int(x2 + pad_x))
    top, bottom = max(0, int(y1 - pad_y)), min(height, int(y2 + pad_y))
    crop = image[top:bottom, left:right]
    return crop.copy() if crop.size else None

print('Batch configuration ready.')


### 6.1 Test V2: annotated-crop OCR review

Cell ini dapat dilanjutkan setelah interupsi. Anotasi menyediakan lokasi plat, tetapi tidak menyediakan transkripsi karakter. Isi kolom `manual_ground_truth` saat review sebelum menghitung akurasi.

In [ ]:
RUN_TEST_V2 = True
test_v2_csv = BATCH_OUTPUT_DIR / f'test_v2_{TEST_V2_SPLIT}_results.csv'
test_v2_images_dir = TEST_V2_DIR / TEST_V2_SPLIT / 'images'
test_v2_labels_dir = TEST_V2_DIR / TEST_V2_SPLIT / 'labels'
test_v2_paths = sorted(path for path in test_v2_images_dir.iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS)
if TEST_V2_MAX_IMAGES is not None:
    test_v2_paths = test_v2_paths[:TEST_V2_MAX_IMAGES]

test_v2_rows = pd.read_csv(test_v2_csv).fillna('').to_dict('records') if test_v2_csv.is_file() else []
for row in test_v2_rows:
    row['reference_id'] = test_v2_reference_id(Path(row['file']))
    row['manual_ground_truth'] = row.get('manual_ground_truth', '')
    if row.get('status') != 'INVALID_IMAGE_OR_LABEL':
        row['status'] = 'VALID_FORMAT' if PLATE_PATTERN.fullmatch(normalize_plate_text(row.get('prediction', ''))) else 'INVALID_FORMAT'
    row.pop('ground_truth', None)
    row.pop('exact_match', None)
    row.pop('character_similarity', None)
if test_v2_rows:
    pd.DataFrame(test_v2_rows).to_csv(test_v2_csv, index=False)
completed_v2 = {row['file'] for row in test_v2_rows}
pending_v2 = [path for path in test_v2_paths if path.name not in completed_v2]
print(f'Test V2 selected={len(test_v2_paths)}, completed={len(completed_v2)}, pending={len(pending_v2)}')

if RUN_TEST_V2:
    for index, image_path in enumerate(pending_v2, start=1):
        image = cv2.imread(str(image_path))
        label_path = test_v2_labels_dir / f'{image_path.stem}.txt'
        crop = crop_from_yolo_polygon(image, label_path) if image is not None else None
        reference_id = test_v2_reference_id(image_path)
        if crop is None:
            row = {'file': image_path.name, 'image_path': str(image_path), 'reference_id': reference_id, 'manual_ground_truth': '', 'prediction': '', 'confidence': 0.0, 'variant': '', 'latency_ms': 0.0, 'status': 'INVALID_IMAGE_OR_LABEL'}
        else:
            prediction, candidates = recognize_crop_for_batch(crop)
            winner = max(candidates, key=lambda item: item['score'])
            status = 'VALID_FORMAT' if PLATE_PATTERN.fullmatch(prediction) else 'INVALID_FORMAT'
            row = {'file': image_path.name, 'image_path': str(image_path), 'reference_id': reference_id, 'manual_ground_truth': '', 'prediction': prediction, 'confidence': winner['confidence'], 'variant': winner['variant'], 'latency_ms': sum(item['latency_ms'] for item in candidates), 'status': status}
        test_v2_rows.append(row)
        if index % CHECKPOINT_EVERY == 0 or index == len(pending_v2):
            pd.DataFrame(test_v2_rows).to_csv(test_v2_csv, index=False)
            print(f'[{index}/{len(pending_v2)}] checkpoint: {test_v2_csv.name}')

test_v2_df = pd.DataFrame(test_v2_rows)
if not test_v2_df.empty:
    print(f'Valid-format prediction rate: {test_v2_df["status"].eq("VALID_FORMAT").mean():.2%}')
    print(f'Median OCR time: {test_v2_df["latency_ms"].median():.1f} ms/image')
    display(test_v2_df.groupby(['variant', 'status']).size().rename('images').reset_index())
    display(test_v2_df[['file', 'reference_id', 'manual_ground_truth', 'prediction', 'confidence', 'variant', 'status']].head(50))
    print('Accuracy belum dihitung: isi manual_ground_truth terlebih dahulu.')


### 6.2 Test V1: end-to-end manual review

Karena tidak ada ground truth pada nama file, hasil ini mengukur coverage deteksi dan menyediakan prediksi untuk direview. Satu baris dibuat untuk setiap plat yang terdeteksi; gambar tanpa deteksi tetap tercatat.

In [ ]:
RUN_TEST_V1 = True
test_v1_csv = BATCH_OUTPUT_DIR / 'test_v1_end_to_end_results.csv'
test_v1_paths = sorted(path for path in TEST_V1_DIR.iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS)
if TEST_V1_MAX_IMAGES is not None:
    test_v1_paths = test_v1_paths[:TEST_V1_MAX_IMAGES]

test_v1_rows = pd.read_csv(test_v1_csv).fillna('').to_dict('records') if test_v1_csv.is_file() else []
for row in test_v1_rows:
    if row['status'] not in {'NO_PLATE_DETECTED', 'INVALID_IMAGE'}:
        row['status'] = 'VALID_FORMAT' if PLATE_PATTERN.fullmatch(normalize_plate_text(row['prediction'])) else 'INVALID_FORMAT'
if test_v1_rows:
    pd.DataFrame(test_v1_rows).to_csv(test_v1_csv, index=False)
completed_v1 = {row['file'] for row in test_v1_rows}
pending_v1 = [path for path in test_v1_paths if path.name not in completed_v1]
print(f'Test V1 selected={len(test_v1_paths)}, completed={len(completed_v1)}, pending={len(pending_v1)}')

if RUN_TEST_V1:
    for index, image_path in enumerate(pending_v1, start=1):
        image = cv2.imread(str(image_path))
        if image is None:
            test_v1_rows.append({'file': image_path.name, 'image_path': str(image_path), 'plate_index': 0, 'prediction': '', 'detection_confidence': 0.0, 'ocr_confidence': 0.0, 'variant': '', 'latency_ms': 0.0, 'status': 'INVALID_IMAGE'})
            continue
        prediction = detector.predict(image, conf=0.35, imgsz=960, device=DEVICE, verbose=False)[0]
        boxes = prediction.boxes if prediction.boxes is not None else []
        if len(boxes) == 0:
            test_v1_rows.append({'file': image_path.name, 'image_path': str(image_path), 'plate_index': 0, 'prediction': '', 'detection_confidence': 0.0, 'ocr_confidence': 0.0, 'variant': '', 'latency_ms': 0.0, 'status': 'NO_PLATE_DETECTED'})
        for plate_index, box in enumerate(boxes, start=1):
            crop = crop_box_with_margin(image, box.xyxy[0].tolist())
            plate_text, candidates = recognize_crop_for_batch(crop)
            winner = max(candidates, key=lambda item: item['score'])
            status = 'VALID_FORMAT' if PLATE_PATTERN.fullmatch(plate_text) else 'INVALID_FORMAT'
            test_v1_rows.append({'file': image_path.name, 'image_path': str(image_path), 'plate_index': plate_index, 'prediction': plate_text, 'detection_confidence': float(box.conf[0]), 'ocr_confidence': winner['confidence'], 'variant': winner['variant'], 'latency_ms': sum(item['latency_ms'] for item in candidates), 'status': status})
        if index % CHECKPOINT_EVERY == 0 or index == len(pending_v1):
            pd.DataFrame(test_v1_rows).to_csv(test_v1_csv, index=False)
            print(f'[{index}/{len(pending_v1)}] checkpoint: {test_v1_csv.name}')

test_v1_df = pd.DataFrame(test_v1_rows)
if not test_v1_df.empty:
    display(test_v1_df['status'].value_counts().rename_axis('status').reset_index(name='images_or_plates'))
    display(test_v1_df.head(50))
    print(f'Result: {test_v1_csv}')


## Interpretation

Utamakan exact accuracy, lalu character similarity. Confidence OCR bukan probabilitas bahwa seluruh nomor polisi benar. Jika detector sudah konsisten tetapi recognition masih sering tertukar satu karakter, langkah berikutnya adalah melatih recognizer khusus karakter plat Indonesia menggunakan crop ber-GT, bukan menambah epoch detector.